# 容量约束弧路径问题(CARP)

**类别：** 路径

来源：[https://www.hexaly.com/templates/capacitated-arc-routing-problem-carp](https://www.hexaly.com/templates/capacitated-arc-routing-problem-carp)


## 问题描述

**在容量约束弧路径问题(Capacitated Arc Routing Problem, CARP)**中,一组具有相同容量的配送车辆必须为具有已知需求的边提供服务。车辆从一个共同的配送中心出发并返回。每条边恰好由一辆车服务。此外,每辆卡车服务的总需求量不得超过其容量。目标是最小化所行驶的总距离。


### 学习要点

- 添加 list decision variables 以建模每辆卡车的边序列
- 在所有列表变量上添加 `disjoint` 约束
- 使用 `contains` 保证每条必服务边的两个方向中恰好有一个被选择
- 定义 lambda 函数 来计算行驶距离


## 数据

所提供的容量约束弧路径问题(CARP)算例来自 [DIMACS 网站](http://dimacs.rutgers.edu/programs/challenge/vrp/carp/)。数据文件的格式如下:

- 节点数量
- 必服务边的数量(具有正需求的边)
- 非必服务边的数量(具有零需求的边)
- 可用车辆数量
- 卡车容量
- 每条必服务边的需求与费用
- 每条非必服务边的费用
- 配送中心节点的索引


## 建模方法

容量约束弧路径问题(CARP)的 OptAgent 模型使用列表变量来表示分配给每辆卡车的边序列。这些边是卡车访问并提供服务的边。卡车在其路径上也可能仅经过其他边而不提供服务。

边可以沿两个方向被访问,但其需求只能被满足一次。OptAgent 的 `disjoint` 约束确保每条边(沿任一方向)至多出现在一个列表变量中,而 `contains(edges_sequences, 2i) + contains(edges_sequences, 2i+1) == 1` 保证每对相反方向中恰好有一条被服务。


`contains(edges_sequences, edge)` 判断某个方向的弧是否出现在任意车辆路线中。

我们可以使用需求数组上的 **at** 算子来访问序列中每条边的需求。我们使用一个 lambda 函数，通过 **sum** 算子对所有被访问边的需求求和，从而计算每辆卡车的总配送量。该总配送量必须不超过卡车的容量。

类似地，我们使用二维距离矩阵上的 **at** 算子来访问从一条边到下一条边所行驶的距离。


## Python 实现


In [ ]:
from optagent import ModelBuilder, solve


def build_carp_model(
    nb_required_edges,
    nb_trucks,
    truck_capacity,
    costs_data,
    demands_data,
    edges_dist_data,
    dist_from_depot_data,
    dist_to_depot_data,
):
    """Construct the CARP model with directed-edge list variables."""
    model = ModelBuilder()

    # Build a feasible initial assignment using the direct orientation of each edge.
    default_sequences = [[] for _ in range(nb_trucks)]
    default_loads = [0] * nb_trucks
    edge_order = sorted(
        range(nb_required_edges),
        key=lambda edge: demands_data[2 * edge],
        reverse=True,
    )
    for edge in edge_order:
        demand = demands_data[2 * edge]
        candidates = [
            truck
            for truck in range(nb_trucks)
            if default_loads[truck] + demand <= truck_capacity
        ]
        if not candidates:
            raise ValueError("Unable to build a capacity-feasible initial assignment")
        truck = min(candidates, key=default_loads.__getitem__)
        default_sequences[truck].append(2 * edge)
        default_loads[truck] += demand

    # Each item is one orientation of a required edge: 2*i is direct, 2*i+1 reverse.
    edges_sequences_vars = [
        model.list(
            2 * nb_required_edges,
            default=tuple(default_sequences[truck]),
            name=f"edges_{truck}",
        )
        for truck in range(nb_trucks)
    ]
    edges_sequences = model.array(edges_sequences_vars)
    model.constraint(model.disjoint(edges_sequences), name="disjoint_edges")

    for edge in range(nb_required_edges):
        model.constraint(
            model.contains(edges_sequences, 2 * edge)
            + model.contains(edges_sequences, 2 * edge + 1)
            == 1,
            name=f"serve_edge_{edge}",
        )

    costs_array = model.array(costs_data)
    demands_array = model.array(demands_data)
    dist_from_depot_array = model.array(dist_from_depot_data)
    dist_to_depot_array = model.array(dist_to_depot_data)
    edges_dist_array = model.array(edges_dist_data)

    route_distances = []
    for k in range(nb_trucks):
        sequence = edges_sequences_vars[k]
        c = model.count(sequence)

        # Truck capacity.
        demand_lambda = model.lambda_function(lambda edge: demands_array[edge])
        route_quantity = model.sum(sequence, demand_lambda)
        model.constraint(route_quantity <= truck_capacity, name=f"cap_{k}")

        # Service cost plus travel from the preceding serviced directed edge.
        dist_lambda = model.lambda_function(
            lambda i: costs_array[sequence[i]]
            + edges_dist_array[sequence[i - 1], sequence[i]]
        )
        route_dist = model.sum(model.range(1, c), dist_lambda) + model.iif(
            c > 0,
            costs_array[sequence[0]]
            + dist_from_depot_array[sequence[0]]
            + dist_to_depot_array[sequence[c - 1]],
            0,
        )
        route_distances.append(route_dist)

    total_distance = model.sum(*route_distances)
    model.minimize(total_distance, name="total_distance")
    return model, edges_sequences_vars


def main():
    # Toy instance: 7 required edges, 3 trucks
    nb_required_edges = 7
    nb_trucks = 3
    truck_capacity = 10

    edge_costs = [10, 20, 30, 40, 50, 60, 70]
    edge_demands = [1, 2, 3, 4, 5, 6, 7]
    edge_endpoints = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 1)]
    costs_data = [cost for cost in edge_costs for _ in range(2)]
    demands_data = [demand for demand in edge_demands for _ in range(2)]
    origins = [node for origin, destination in edge_endpoints for node in (origin, destination)]
    destinations = [node for origin, destination in edge_endpoints for node in (destination, origin)]
    edges_dist_data = [
        [0 if destinations[i] == origins[j] else abs(destinations[i] - origins[j]) * 5
         for j in range(2 * nb_required_edges)]
        for i in range(2 * nb_required_edges)
    ]
    dist_from_depot_data = [abs(origin - 1) * 5 for origin in origins]
    dist_to_depot_data = [abs(destination - 1) * 5 for destination in destinations]

    model, edges_sequences_vars = build_carp_model(
        nb_required_edges=nb_required_edges,
        nb_trucks=nb_trucks,
        truck_capacity=truck_capacity,
        costs_data=costs_data,
        demands_data=demands_data,
        edges_dist_data=edges_dist_data,
        dist_from_depot_data=dist_from_depot_data,
        dist_to_depot_data=dist_to_depot_data,
    )

    solution = solve(model, time_limit_s=10.0)
    print(f"status: {solution.status.value}")
    print(f"objective: {solution.objective_value}")
    for k in range(nb_trucks):
        seq = solution.variable_values[edges_sequences_vars[k].node_id]
        if seq:
            serviced = [(origins[edge], destinations[edge]) for edge in seq]
            print(f"Truck {k}: {serviced}")


if __name__ == '__main__':
    main()


Starting OptAgent PORTFOLIO
Parameters: time_limit=10s threads=auto seed=0
